# Resume Classification (NLP)
This project categorizes resumes into various job roles using NLP techniques and a Support Vector Machine (SVM) classifier.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import pickle
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

## Load Dataset

In [ ]:
try:
    df = pd.read_csv('UpdatedResumeDataSet.csv')
    print(f"Dataset loaded. Shape: {df.shape}")
    print(df.head())
except FileNotFoundError:
    print("UpdatedResumeDataSet.csv not found.")

## Data Cleaning

In [ ]:
def clean_resume(txt):
    txt = re.sub('http\S+\s', ' ', txt)
    txt = re.sub('RT|cc', ' ', txt)
    txt = re.sub('#\S+\s', ' ', txt)
    txt = re.sub('@\S+', '  ', txt)
    txt = re.sub('[%s]' % re.escape("""!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~"""), ' ', txt)
    txt = re.sub(r'[^\x00-\x7f]', ' ', txt)
    txt = re.sub('\s+', ' ', txt)
    return txt.strip()

df['Resume_Cleaned'] = df['Resume'].apply(clean_resume)

## Label Encoding and Vectorization

In [ ]:
le = LabelEncoder()
df['Category_Encoded'] = le.fit_transform(df['Category'])

tfidf = TfidfVectorizer(stop_words='english')
X = tfidf.fit_transform(df['Resume_Cleaned'])
y = df['Category_Encoded']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Model Training

In [ ]:
model = OneVsRestClassifier(SVC())
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
pickle.dump(tfidf, open('tfidf_resume.pkl', 'wb'))
pickle.dump(model, open('resume_model.pkl', 'wb'))
pickle.dump(le, open('encoder_resume.pkl', 'wb'))